# Cross-model generalization: does a ChatGPT-trained classifier detect Claude text?

The model in `results/checkpoints/length_controlled/final` was trained only on
HC3 (human vs. **ChatGPT**, reddit_eli5 Q&A domain). Here we test it on a
completely different source: the **Claude**-generated essay subset of the
Kaggle DAIGT V2 dataset (`darragh_claude_v6` / `darragh_claude_v7`, ~2000
essays by Claude), paired against the human student essays in the same
dataset (`persuade_corpus`) that those Claude essays were written to match.

**Important confound, by design (see project decision log)**: this swaps
*both* the generator model (ChatGPT -> Claude) *and* the domain (Reddit Q&A ->
persuasive student essays) at once. A drop in accuracy here could be due to
either shift, not model-shift alone — we report this honestly as a limitation
rather than claiming a clean model-generalization result.

**Prerequisites**:
1. The `length_controlled` checkpoint downloaded locally (see `04_interpretability.ipynb`).
2. A Kaggle account with API credentials configured, either via
   `~/.kaggle/kaggle.json` or by running `kagglehub.login()` in the cell below.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import DistilBertForSequenceClassification

from model import load_tokenizer

CHECKPOINT_DIR = Path.cwd().parent / 'results' / 'checkpoints' / 'length_controlled' / 'final'
assert CHECKPOINT_DIR.exists(), f'{CHECKPOINT_DIR} not found — download it from Colab/Drive first (see 03_finetune_distilbert.ipynb).'

tokenizer = load_tokenizer(str(CHECKPOINT_DIR))
model = DistilBertForSequenceClassification.from_pretrained(str(CHECKPOINT_DIR))
model.eval()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

## Download DAIGT V2 and pull out the Claude / human-essay subset

`thedrcat/daigt-v2-train-dataset` (44,868 rows total) contains a `source`
column identifying the generator: `persuade_corpus` for the original human
student essays, `darragh_claude_v6` / `darragh_claude_v7` for ~1000 Claude
essays each written against the same PERSUADE prompts.

In [2]:
import kagglehub

# uncomment if kagglehub can't find credentials automatically:
# kagglehub.login()

daigt_path = kagglehub.dataset_download('thedrcat/daigt-v2-train-dataset')
csv_path = Path(daigt_path) / 'train_v2_drcat_02.csv'
daigt_df = pd.read_csv(csv_path)
print(daigt_df.shape)
daigt_df['source'].value_counts()

  0%|          | 0.00/28.5M [00:00<?, ?B/s]

  4%|▎         | 1.00M/28.5M [00:00<00:14, 1.93MB/s]

 11%|█         | 3.00M/28.5M [00:00<00:05, 5.05MB/s]

 28%|██▊       | 8.00M/28.5M [00:00<00:01, 14.5MB/s]

 39%|███▊      | 11.0M/28.5M [00:01<00:01, 13.3MB/s]

...

 98%|█████████▊| 28.0M/28.5M [00:01<00:00, 24.6MB/s]

100%|██████████| 28.5M/28.5M [00:01<00:00, 15.2MB/s]

Extracting files...


(44868, 5)


source
persuade_corpus                       25996
mistral7binstruct_v2                   2421
chat_gpt_moth                          2421
mistral7binstruct_v1                   2421
llama2_chat                            2421
kingki19_palm                          1384
train_essays                           1378
llama_70b_v1                           1172
falcon_180b_v1                         1055
darragh_claude_v7                      1000
darragh_claude_v6                      1000
radek_500                               500
NousResearch/Llama-2-7b-chat-hf         400
mistralai/Mistral-7B-Instruct-v0.1      400
cohere-command                          350
palm-text-bison1                        349
radekgpt4                               200
Name: count, dtype: int64

In [3]:
CLAUDE_SOURCES = ['darragh_claude_v6', 'darragh_claude_v7']
HUMAN_SOURCE = 'persuade_corpus'

claude_df = daigt_df[daigt_df['source'].isin(CLAUDE_SOURCES)][['text', 'label']].copy()
human_df = daigt_df[daigt_df['source'] == HUMAN_SOURCE][['text', 'label']].copy()

# balance classes: subsample human essays down to the Claude subset size
human_df = human_df.sample(n=len(claude_df), random_state=42)

cross_df = pd.concat([claude_df, human_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
print('claude (label should be 1):', claude_df['label'].unique())
print('human (label should be 0):', human_df['label'].unique())
print('cross_df shape:', cross_df.shape, 'balance:', cross_df['label'].value_counts().to_dict())

claude (label should be 1): [1]
human (label should be 0): [0]
cross_df shape: (4000, 2) balance: {1: 2000, 0: 2000}


## Evaluate the ChatGPT-trained model on the Claude/human-essay set

In [4]:
@torch.no_grad()
def predict_labels(model, tokenizer, texts, max_length=256, batch_size=16):
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = list(texts[i:i + batch_size])
        enc = tokenizer(batch, truncation=True, padding=True, max_length=max_length, return_tensors='pt')
        logits = model(**enc).logits
        preds.extend(torch.argmax(logits, dim=-1).tolist())
    return preds


cross_df['pred'] = predict_labels(model, tokenizer, cross_df['text'])

precision, recall, f1, _ = precision_recall_fscore_support(cross_df['label'], cross_df['pred'], average='binary')
acc = accuracy_score(cross_df['label'], cross_df['pred'])
cross_metrics = {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}
print('Cross-model (Claude essays) metrics:', cross_metrics)

Cross-model (Claude essays) metrics: {'accuracy': 0.75525, 'precision': 0.8036882807852469, 'recall': 0.6755, 'f1': 0.7340396631350177}


## Compare against in-domain (HC3/ChatGPT) test performance

Loads the same model's held-out HC3 test set (saved during training) to show
the in-domain vs. cross-model/cross-domain gap side by side.

In [5]:
hc3_test_df = pd.read_parquet(CHECKPOINT_DIR / 'test_split.parquet')
hc3_test_df['pred'] = predict_labels(model, tokenizer, hc3_test_df['text'])

precision, recall, f1, _ = precision_recall_fscore_support(hc3_test_df['label'], hc3_test_df['pred'], average='binary')
acc = accuracy_score(hc3_test_df['label'], hc3_test_df['pred'])
in_domain_metrics = {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

comparison = pd.DataFrame([
    {'setting': 'in-domain (HC3, ChatGPT, reddit_eli5)', **in_domain_metrics},
    {'setting': 'cross-model+domain (DAIGT, Claude, essays)', **cross_metrics},
])
comparison

,setting,accuracy,precision,recall,f1
0,"in-domain (HC3, ChatGPT, reddit_eli5)",0.986395,0.975362,0.997999,0.986551
1,"cross-model+domain (DAIGT, Claude, essays)",0.755250,0.803688,0.675500,0.734040


In [6]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(cross_df['label'], cross_df['pred'])
print('confusion matrix (rows=true, cols=pred), [0=human, 1=ai]:')
print(cm)
print(f"\nhuman-essay recall (specificity): {cm[0, 0] / cm[0].sum():.3f}")
print(f"claude-essay recall (sensitivity): {cm[1, 1] / cm[1].sum():.3f}")

import json
results_dir = Path.cwd().parent / 'results'
with open(results_dir / 'metrics.json', 'a', encoding='utf-8') as f:
    f.write(json.dumps({'cross_model_test': comparison.to_dict(orient='records')}) + '\n')

confusion matrix (rows=true, cols=pred), [0=human, 1=ai]:
[[1670  330]
 [ 649 1351]]

human-essay recall (specificity): 0.835
claude-essay recall (sensitivity): 0.675


## Interpretation notes for the write-up

- Any accuracy drop here reflects **domain shift (Q&A -> essays) and model
  shift (ChatGPT -> Claude) combined** — this setup cannot isolate which one
  drives it. Say so explicitly rather than calling this a clean
  "cross-model" result.
- If specificity (human-essay recall) holds but sensitivity (Claude recall)
  drops, that's consistent with the model's learned "AI style" being
  ChatGPT-specific rather than a general AI-text detector.
- Worth cross-referencing with `04_interpretability.ipynb`: if the
  length_controlled model's top attributed tokens are HC3/Reddit-register
  words (contractions, informal phrasing) rather than general AI-generation
  markers, that's independent evidence the model learned something
  domain-specific, not just "AI-ness."